# Агент "Куда сходить в Алматы"

ТЗ #2: парсинг sxodim.com/almaty + агент, отвечающий на вопросы о том, куда сходить.

## Раздел 1: Парсинг сайта

Скрейпинг запускается один раз и кэшируется в `sxodim_raw.json` — при повторном запуске ноутбука сеть не дёргается, если файл уже существует. Логика парсинга живёт в `scraper.py` (см. рядом), ноутбук только вызывает её и показывает результат.

**Объём данных — сознательное решение, не случайный обрыв.** События (`/almaty/afisha?page=1..9`) берутся полностью — сайт отдаёт все 139 событий за 9 страниц. Места устроены иначе: общий листинг `/almaty/places` — это 119 страниц (~2260 мест), сильно перекошенных по категориям (одни только кафе — 77 страниц). Чтобы не терять маленькие, но важные для вопросов ТЗ категории (например, `dlja-detej` — «для детей» — всего 2 страницы) в шуме больших, мы берём **первые 2 страницы каждой из 22 категорий мест** (`places/<category>?page=1,2`), а не первые N страниц общего листинга. Это даёт сбалансированный срез по всем типам мест вместо перекоса в сторону кафе/ресторанов — см. `docs/superpowers/specs/2026-09-28-sxodim-agent-design.md`, раздел 4.1.

In [1]:
import json
from pathlib import Path

import requests

import scraper

RAW_PATH = Path("sxodim_raw.json")

In [2]:
if RAW_PATH.exists():
    records = json.loads(RAW_PATH.read_text(encoding="utf-8"))
    print(f"Loaded {len(records)} cached records from {RAW_PATH}")
else:
    session = requests.Session()
    fetch = scraper.make_fetcher(session)
    records = scraper.scrape_all(fetch)
    scraper.save_raw(records, str(RAW_PATH))
    print(f"Scraped {len(records)} records -> {RAW_PATH}")

Loaded 592 cached records from sxodim_raw.json


In [3]:
events = [r for r in records if r["type"] == "event"]
places = [r for r in records if r["type"] == "place"]
print(f"events: {len(events)}, places: {len(places)}, total: {len(records)}")

events: 136, places: 456, total: 592


In [4]:
print("Пример события:")
print(json.dumps(events[0], ensure_ascii=False, indent=2))
print()
print("Пример места:")
print(json.dumps(places[0], ensure_ascii=False, indent=2))

Пример события:
{
  "type": "event",
  "url": "https://sxodim.com/almaty/event/abzal-teshovty-koncerti",
  "name": "Абзал Утешов Алматыда",
  "category": "Концерты во Дворце Республики",
  "description": "Абзал Утешов – қазақстандық эстраданың жас әрі жарқын өкілдерінің бірі. Ол тыңдармандардың жүрегінен орын алған әсерлі әндері мен шынайы орындау мәнері арқылы кеңінен танымал болды.",
  "address": "Республика сарайы, Достық даңғылы, 56",
  "date_start": "2026-10-07T20:00:00+05:00",
  "price": 11000,
  "currency": "KZT",
  "rating": null,
  "image": "https://sxodim.com/uploads/posts/2026/08/02/optimized/d0fcc3bb75454a3f7c4f79e31ae74adb_1522x570-q-85.jpg"
}

Пример места:
{
  "type": "place",
  "url": "https://sxodim.com/almaty/place/17st-lounge-bar",
  "name": "17st. Lounge Bar",
  "category": "Бары",
  "description": "Посетители смогут окунуться в атмосферу вкусных стейков, крафтового пенного и дымных сетов.",
  "address": "17st. Lounge Bar, ул. Розыбакиева, 166",
  "date_start": null

## Раздел 2: Структурирование данных

Дальше — сырые записи из `sxodim_raw.json` превращаются в финальный `sxodim_data.json`: каждая запись получает стабильный `id` и список `tags`. Теги выводятся **детерминированно** из категории (подстрочные правила вида "категория содержит «концерт»" → тег `концерт`) и из цены (`price == 0` → `бесплатно`) — без обращения к LLM. Логика — в `normalize.py`.

In [5]:
import normalize

data = normalize.normalize_all(records)
normalize.save_data(data, "sxodim_data.json")
print(f"Normalized {len(data)} records -> sxodim_data.json")

Normalized 592 records -> sxodim_data.json


In [6]:
import collections

tag_counts = collections.Counter(t for r in data for t in r["tags"])
no_tags = sum(1 for r in data if not r["tags"])
print(f"tag distribution: {tag_counts.most_common()}")
print(f"records with no tags: {no_tags} (categories: {sorted({r['category'] for r in data if not r['tags']})})")

tag distribution: [('активный_отдых', 193), ('искусство', 92), ('еда', 90), ('ночная_жизнь', 78), ('образование', 42), ('концерт', 33), ('для_детей', 19), ('романтика', 11)]
records with no tags: 49 (categories: ['Коворкинги', 'Красота и здоровье', 'Отели'])


In [7]:
sample_event = next(r for r in data if r["type"] == "event")
sample_place = next(r for r in data if r["type"] == "place")
print("Структурированное событие:")
print(json.dumps(sample_event, ensure_ascii=False, indent=2))
print()
print("Структурированное место:")
print(json.dumps(sample_place, ensure_ascii=False, indent=2))

Структурированное событие:
{
  "type": "event",
  "url": "https://sxodim.com/almaty/event/abzal-teshovty-koncerti",
  "name": "Абзал Утешов Алматыда",
  "category": "Концерты во Дворце Республики",
  "description": "Абзал Утешов – қазақстандық эстраданың жас әрі жарқын өкілдерінің бірі. Ол тыңдармандардың жүрегінен орын алған әсерлі әндері мен шынайы орындау мәнері арқылы кеңінен танымал болды.",
  "address": "Республика сарайы, Достық даңғылы, 56",
  "date_start": "2026-10-07T20:00:00+05:00",
  "price": 11000,
  "currency": "KZT",
  "rating": null,
  "image": "https://sxodim.com/uploads/posts/2026/08/02/optimized/d0fcc3bb75454a3f7c4f79e31ae74adb_1522x570-q-85.jpg",
  "id": "event-abzal-teshovty-koncerti",
  "tags": [
    "концерт"
  ]
}

Структурированное место:
{
  "type": "place",
  "url": "https://sxodim.com/almaty/place/17st-lounge-bar",
  "name": "17st. Lounge Bar",
  "category": "Бары",
  "description": "Посетители смогут окунуться в атмосферу вкусных стейков, крафтового пенного

Дальше (следующий этап, отдельным пайплайном): сам агент — фильтрация по `tags`/`category`/`price`/`date_start` + генерация ответа — см. `docs/superpowers/specs/2026-09-28-sxodim-agent-design.md`.